# s01 — Fetch neuropil and neuron meshes

Downloads, from FlyWire, into `Processed_Data/`:

- the whole-brain background mesh → `Processed_Data/fig1a_brain_mesh/`
- the optic-lobe neuropil region meshes (Me / Lo / LoP / AMe, left and right), plus the
  combined whole-optic-lobe meshes (Optic_R / Optic_L = Me + Lo + LoP + AMe)
  → `Processed_Data/optic_lobe_neuropil_mesh/`
- the neuron meshes for Fig 1A, Fig 1B, Fig 1H and Fig 5A → their respective
  `Processed_Data/fig1*_neuron_mesh/` and `Processed_Data/fig5a_neuron_mesh/` folders

The optic-lobe neuropil meshes are read by `Data_Processing/s11_FBP_output_layer.m`
(reference-layer / target neuropils); the combined Optic_R mesh is read by
`Figures/fig_5I_J_LC9_interconnection.m`. They are kept out of `fig1a_brain_mesh/` so the
`Figures/fig_1A_neurons.m` background stays the whole-brain mesh only.

In [ ]:
from fafbseg import flywire
import navis
import numpy as np
import os

# baseDir is resolved from the notebook's location (run from Data_Processing/),
# so the code runs after download without editing any path.
baseDir = os.path.dirname(os.getcwd())
processed_dir = os.path.join(baseDir, "Processed_Data")

In [ ]:
# pip install cloud-volume numpy
from cloudvolume import CloudVolume

# Output folder (background brain mesh for fig_1A). Read by Figures/fig_1A_neurons.m -> neuropilDir
out_dir = os.path.join(processed_dir, "fig1a_brain_mesh")
os.makedirs(out_dir, exist_ok=True)

# Codex/Neuroglancer precomputed source
cloudpath = "precomputed://gs://flywire_neuropil_meshes/whole_neuropil/brain_mesh_v3"

# use_https=True can help work around gs:// access issues in some environments.
cv = CloudVolume(cloudpath, use_https=True, progress=True)

# The brain mesh is usually stored under a low segment id (exact id mapping
# depends on the view state / documentation).
# Try id 1 first (if it fails, use a scan over ids).
seg_id = 1

mesh = cv.mesh.get(seg_id)  # Mesh object
V = np.asarray(mesh.vertices)          # (N,3) float
F = np.asarray(mesh.faces)             # (M,3) int (usually 0-based)

np.savetxt(os.path.join(out_dir, "brain_mesh_v3_vertices.csv"), V, delimiter=",")
np.savetxt(os.path.join(out_dir, "brain_mesh_v3_faces.csv"), F, delimiter=",", fmt="%d")
print("saved:", V.shape, F.shape)

In [ ]:
# Download the neuron meshes shown in Figure 1A
from cloudvolume import CloudVolume

# Output folder (foreground neuron meshes for fig_1A). Read by Figures/fig_1A_neurons.m -> neuronDir
out_dir = os.path.join(processed_dir, "fig1a_neuron_mesh")
os.makedirs(out_dir, exist_ok=True)

cloudpath = "precomputed://gs://flywire_v141_m783"

# use_https=True can help work around gs:// access issues in some environments.
cv = CloudVolume(cloudpath, use_https=True, progress=True)

# root_ids of the neurons shown in Figure 1A
seg_ids = [
    720575940611633010,  # LT51
    720575940627706398,  # VCH
    720575940627758744,  # LC14
]

for seg_id in seg_ids:
    meshes = cv.mesh.get(seg_id)      # returns a dict keyed by seg_id

    # 1. Extract the Mesh object for this id from the dict
    target_mesh = meshes[seg_id]

    # 2. Get vertices and faces from the Mesh object
    V = np.asarray(target_mesh.vertices)
    F = np.asarray(target_mesh.faces)

    np.savetxt(os.path.join(out_dir, f"{seg_id}_vertices.csv"), V, delimiter=",")
    np.savetxt(os.path.join(out_dir, f"{seg_id}_faces.csv"), F, delimiter=",", fmt="%d")

    print(f"Saved {seg_id}: Vertices {V.shape}, Faces {F.shape}")

In [ ]:
# Download the LC9 neuron mesh shown in Figure 1B
# Output folder. Read by Figures/fig_1B_LC9.m
out_dir = os.path.join(processed_dir, "fig1b_neuron_mesh")
os.makedirs(out_dir, exist_ok=True)

# reuse cv (precomputed://gs://flywire_v141_m783) from the previous cell
seg_id = 720575940635224943  # LC9 (Figure 1B)

meshes = cv.mesh.get(seg_id)      # returns a dict keyed by seg_id
target_mesh = meshes[seg_id]
V = np.asarray(target_mesh.vertices)
F = np.asarray(target_mesh.faces)

np.savetxt(os.path.join(out_dir, f"{seg_id}_vertices.csv"), V, delimiter=",")
np.savetxt(os.path.join(out_dir, f"{seg_id}_faces.csv"), F, delimiter=",", fmt="%d")

print(f"Saved {seg_id}: Vertices {V.shape}, Faces {F.shape}")

In [ ]:
# Download the neuron meshes shown in Figure 1H
# Output folder. Read by Figures/fig_1H_neurons_specific_angle.m
out_dir = os.path.join(processed_dir, "fig1h_neuron_mesh")
os.makedirs(out_dir, exist_ok=True)

# reuse cv (precomputed://gs://flywire_v141_m783) from a previous cell
seg_ids = [
    720575940623047629,  # LPLC2
    720575940642423797,  # cLP01
    720575940635052569,  # LT52
    720575940628406538,  # LT58
    720575940633124143,  # PLP215
]

for seg_id in seg_ids:
    meshes = cv.mesh.get(seg_id)      # returns a dict keyed by seg_id
    target_mesh = meshes[seg_id]
    V = np.asarray(target_mesh.vertices)
    F = np.asarray(target_mesh.faces)

    np.savetxt(os.path.join(out_dir, f"{seg_id}_vertices.csv"), V, delimiter=",")
    np.savetxt(os.path.join(out_dir, f"{seg_id}_faces.csv"), F, delimiter=",", fmt="%d")

    print(f"Saved {seg_id}: Vertices {V.shape}, Faces {F.shape}")

In [ ]:
# Download the optic-lobe neuropil region meshes.
# Kept in a dedicated folder (NOT fig1a_brain_mesh) so the fig_1A background stays the
# whole-brain mesh only. Read by Data_Processing/s11_FBP_output_layer.m -> neuropilDir.
out_dir = os.path.join(processed_dir, "optic_lobe_neuropil_mesh")
os.makedirs(out_dir, exist_ok=True)

# map: FlyWire neuropil id -> output filename prefix (matches what the .m scripts read)
neuropils = {
    "ME_L": "Me_L",   "ME_R": "Me_R",
    "LO_L": "Lo_L",   "LO_R": "Lo_R",
    "LOP_L": "LoP_L", "LOP_R": "LoP_R",
    "AME_L": "AMe_L", "AME_R": "AMe_R",
}

for npil_id, prefix in neuropils.items():
    vol = flywire.get_neuropil_volumes(npil_id)
    V = np.asarray(vol.vertices)
    F = np.asarray(vol.faces)
    np.savetxt(os.path.join(out_dir, f"{prefix}_vertices.csv"), V, delimiter=",")
    np.savetxt(os.path.join(out_dir, f"{prefix}_faces.csv"), F, delimiter=",", fmt="%d")
    print(f"Saved {npil_id} -> {prefix}: Vertices {V.shape}, Faces {F.shape}")

In [ ]:
# Build combined whole-optic-lobe meshes (right and left) by merging the per-region
# volumes (Me + Lo + LoP + AMe). Read by Figures/fig_5I_J_LC9_interconnection.m (Optic_R).
out_dir = os.path.join(processed_dir, "optic_lobe_neuropil_mesh")
os.makedirs(out_dir, exist_ok=True)

Me_R  = flywire.get_neuropil_volumes("ME_R")
Lo_R  = flywire.get_neuropil_volumes("LO_R")
LoP_R = flywire.get_neuropil_volumes("LOP_R")
AMe_R = flywire.get_neuropil_volumes("AME_R")
Me_L  = flywire.get_neuropil_volumes("ME_L")
Lo_L  = flywire.get_neuropil_volumes("LO_L")
LoP_L = flywire.get_neuropil_volumes("LOP_L")
AMe_L = flywire.get_neuropil_volumes("AME_L")

Optic_R = navis.Volume.combine([Me_R, Lo_R, LoP_R, AMe_R])
Optic_L = navis.Volume.combine([Me_L, Lo_L, LoP_L, AMe_L])

for vol, prefix in [(Optic_R, "Optic_R"), (Optic_L, "Optic_L")]:
    V = np.asarray(vol.vertices)
    F = np.asarray(vol.faces)
    np.savetxt(os.path.join(out_dir, f"{prefix}_vertices.csv"), V, delimiter=",")
    np.savetxt(os.path.join(out_dir, f"{prefix}_faces.csv"), F, delimiter=",", fmt="%d")
    print(f"Saved {prefix}: Vertices {V.shape}, Faces {F.shape}")

In [ ]:
# Download the neuron meshes shown in Figure 5A
# Output folder. Read by Figures/fig_5A_BDP_neurons.m -> meshDir
out_dir = os.path.join(processed_dir, "fig5a_neuron_mesh")
os.makedirs(out_dir, exist_ok=True)

# reuse cv (precomputed://gs://flywire_v141_m783) from a previous cell
seg_ids = [
    720575940635224943,  # LC9
    720575940629376528,  # LT43
    720575940631938647,  # LT52
]

for seg_id in seg_ids:
    meshes = cv.mesh.get(seg_id)      # returns a dict keyed by seg_id
    target_mesh = meshes[seg_id]
    V = np.asarray(target_mesh.vertices)
    F = np.asarray(target_mesh.faces)

    np.savetxt(os.path.join(out_dir, f"{seg_id}_vertices.csv"), V, delimiter=",")
    np.savetxt(os.path.join(out_dir, f"{seg_id}_faces.csv"), F, delimiter=",", fmt="%d")

    print(f"Saved {seg_id}: Vertices {V.shape}, Faces {F.shape}")